# 03 — Gold: Scheduling Reliability & Cost Exposure


**Reads:** `silver.patient_information` — 64,353 rows, 39 columns  
**Writes:** `gold.procedure_baseline` — 418 procedures; `gold.fact_cases` — 48,118 cases, 26 columns

### Business problem

OR schedules depend on reliable procedure-duration estimates. When cases run substantially
longer than expected, they can disrupt downstream scheduling. When they finish substantially
earlier, allocated OR time may go unused. This Gold layer uses historical procedure duration
as a scheduling benchmark to quantify that uncertainty.

### Business questions

1. **How reliably do actual OR case durations align with historical procedure-duration baselines?**
2. **How often do cases finish more than 30 or 60 minutes above baseline, or more than 30 minutes below it?**
3. **What is the estimated annual cost exposure associated with these deviations, under different cost-per-minute assumptions?**
4. **Which procedures should be prioritized for review because their historical duration estimates are least reliable?**

### Scope & limitations

`expected_duration_min` is the historical median OR duration for each procedure, not an
originally scheduled time; the source data carries no original schedule to compare against.
Only procedures with at least 30 qualifying cases are included in the analytical cohort.

This notebook identifies scheduling opportunities and estimated cost exposure. It does not
claim scheduling performance or costs improved after an intervention, since the dataset has no
post-change period to measure against.

Headline figures were checked for robustness twice: once against a cleaning pass removing
duplicates and repairing corrupt timestamps, and again after adding one previously excluded
case. In both checks, MAE, overrun rates, and median absolute error remained essentially
unchanged, indicating that the headline findings are not driven by a small number of
edge-case records.


## Building `gold.procedure_baseline`

One row per procedure, restricted to cases with a usable OR duration (10 to 720 minutes, both
timestamps present) and at least 30 qualifying cases. The `n >= 30` threshold lives here and
only here; everything downstream inherits it through the join rather than re-filtering.

In [4]:
%%sql
-- baseline duration stats, per procedure, cohort filter (n >= 30)
CREATE OR REPLACE TABLE gold.procedure_baseline AS
SELECT
    procedure_nm,
    COUNT(*) AS case_count,
    CAST(percentile(or_duration_min, 0.5) AS DECIMAL(10,1)) AS expected_duration_min,
    CAST(AVG(or_duration_min) AS DECIMAL(10,1)) AS mean_duration_min,
    CAST(STDDEV(or_duration_min) AS DECIMAL(10,2)) AS stdev_duration_min,
    CAST(STDDEV(or_duration_min) / AVG(or_duration_min) AS DECIMAL(10,3)) AS cv,
    CAST(COUNT(*) * AVG(or_duration_min) / 60 AS DECIMAL(12,1)) AS or_hours_consumed
FROM silver.patient_information
WHERE in_or IS NOT NULL
  AND out_or IS NOT NULL
  AND or_duration_min BETWEEN 10 AND 720
GROUP BY procedure_nm
HAVING COUNT(*) >= 30

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Building `gold.fact_cases`

The case-level table is trimmed from Silver's 39 columns to the 20 Gold actually needs.
`asa_rating_c` is dropped because `asa_rating` carries the same information as a readable
label, with identical null counts and a clean one-to-one mapping confirmed during Silver
validation. `out_or`, `an_start`, `an_stop`, `anes_duration_min`, and `or_minus_anes` are also
dropped; they served their purpose in identifying and validating timestamp repairs and are no
longer needed once those repairs have been applied.

Each case is then measured against its procedure baseline. `schedule_error_min` keeps the
direction of the deviation (positive means longer than baseline), while `abs_error_min`
captures its magnitude regardless of direction. Three flags identify cases more than 30 or
60 minutes above baseline and more than 30 minutes below baseline.

In [5]:
# case-level table, trimmed to the 20 columns gold actually needs
drop_cols = [
    "hosp_admsn_time", "hosp_disch_time", "surgery_date",
    "los", "icu_admin_flag", "disch_disp_c",
    "height", "weight", "primary_procedure_nm",
    "in_or_dttm", "out_or_dttm", "an_start_datetime", "an_stop_datetime",
    "asa_rating_c", "out_or", "an_start", "an_stop", "anes_duration_min", "or_minus_anes",
]

s = spark.table("silver.patient_information").drop(*drop_cols)
s.createOrReplaceTempView("patient_information_gold")

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 7, Finished, Available, Finished, False)

In [6]:
%%sql
-- case-level fact table: schedule deviation per case, in both directions
CREATE OR REPLACE TABLE gold.fact_cases AS

SELECT
    s.*,
    b.expected_duration_min,

    CAST(
        s.or_duration_min - b.expected_duration_min
        AS DECIMAL(10,1)
    ) AS schedule_error_min,

    CAST(
        ABS(s.or_duration_min - b.expected_duration_min)
        AS DECIMAL(10,1)
    ) AS abs_error_min,

    s.or_duration_min - b.expected_duration_min > 30 AS overrun_30,
    s.or_duration_min - b.expected_duration_min > 60 AS overrun_60,
    s.or_duration_min - b.expected_duration_min < -30 AS early_30

FROM patient_information_gold s

JOIN gold.procedure_baseline b
    ON s.procedure_nm = b.procedure_nm

WHERE s.in_or IS NOT NULL
  AND s.or_duration_min BETWEEN 10 AND 720;

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Enriching `procedure_baseline` with exposure metrics

Adds MAE, median absolute error, threshold rates, and total deviation minutes per procedure,
computed from `fact_cases`. The table is rebuilt from its original Silver source alongside
the new case-level metrics rather than reading from and replacing itself, keeping the
`CREATE OR REPLACE TABLE` operation safe to rerun without a self-read conflict.

Reliability and exposure are kept as separate measures rather than collapsed into a single
priority score. A common, predictable procedure can accumulate high total exposure through
volume alone, while a lower-volume procedure can be highly unreliable without contributing
as much total exposure. Keeping both dimensions visible preserves that distinction for
procedure prioritization.

In [7]:
%%sql
-- enrich procedure_baseline with exposure metrics. Recomputes baseline stats
-- fresh from silver rather than reading gold.procedure_baseline, so this stays
-- a clean CREATE OR REPLACE with no self-read and no schema-evolution risk on rerun.
CREATE OR REPLACE TABLE gold.procedure_baseline AS

WITH baseline AS (
    SELECT
        procedure_nm,
        COUNT(*) AS case_count,
        CAST(percentile(or_duration_min, 0.5) AS DECIMAL(10,1)) AS expected_duration_min,
        CAST(AVG(or_duration_min) AS DECIMAL(10,1)) AS mean_duration_min,
        CAST(STDDEV(or_duration_min) AS DECIMAL(10,2)) AS stdev_duration_min,
        CAST(STDDEV(or_duration_min) / AVG(or_duration_min) AS DECIMAL(10,3)) AS cv,
        CAST(COUNT(*) * AVG(or_duration_min) / 60 AS DECIMAL(12,1)) AS or_hours_consumed
    FROM silver.patient_information
    WHERE in_or IS NOT NULL
      AND out_or IS NOT NULL
      AND or_duration_min BETWEEN 10 AND 720
    GROUP BY procedure_nm
    HAVING COUNT(*) >= 30
),
exposure AS (
    SELECT
        procedure_nm,
        CAST(AVG(abs_error_min) AS DECIMAL(10,1)) AS mae_min,
        CAST(percentile(abs_error_min, 0.5) AS DECIMAL(10,1)) AS median_abs_error_min,
        CAST(100.0 * AVG(CASE WHEN ABS(schedule_error_min) > 30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_outside_30,
        CAST(100.0 * AVG(CASE WHEN overrun_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_overrun_30,
        CAST(100.0 * AVG(CASE WHEN early_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_early_30,
        CAST(SUM(abs_error_min) AS DECIMAL(12,1)) AS total_deviation_min
    FROM gold.fact_cases
    GROUP BY procedure_nm
)
SELECT b.*, e.mae_min, e.median_abs_error_min, e.pct_outside_30,
       e.pct_overrun_30, e.pct_early_30, e.total_deviation_min
FROM baseline b
JOIN exposure e ON b.procedure_nm = e.procedure_nm

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### Question 1 — How reliably do actual durations align with baseline?

In [8]:
%%sql
SELECT
    COUNT(*) AS cases,
    CAST(AVG(abs_error_min) AS DECIMAL(10,1)) AS mae_min,
    CAST(percentile(abs_error_min, 0.5) AS DECIMAL(10,1)) AS median_abs_error_min
FROM gold.fact_cases

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### Question 2 — How often do cases land outside the 30 and 60 minute bands, in either direction?

In [9]:
%%sql
SELECT
    CAST(100.0 * AVG(CASE WHEN overrun_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_overrun_30,
    CAST(100.0 * AVG(CASE WHEN overrun_60 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_overrun_60,
    CAST(100.0 * AVG(CASE WHEN early_30 THEN 1 ELSE 0 END) AS DECIMAL(5,1)) AS pct_early_30
FROM gold.fact_cases

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

### Question 3 — What is the estimated annual cost exposure?

Two scenarios, $35/min and $60/min, since the true cost-per-minute of OR time is an
assumption, not a fact in this dataset, and a single number would understate that.

In [10]:
%%sql
SELECT
    SUM(abs_error_min) AS total_error_min,
    ROUND(SUM(abs_error_min) / 5.75) AS error_min_per_year,
    ROUND(SUM(abs_error_min) / 5.75 * 35 / 1000000, 1) AS cost_musd_at_35,
    ROUND(SUM(abs_error_min) / 5.75 * 60 / 1000000, 1) AS cost_musd_at_60
FROM gold.fact_cases

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

### Question 4 — Which procedures should be prioritized for review?

Ranked by `total_deviation_min`, each procedure's actual share of the total exposure, with
`mae_min` and `pct_outside_30` shown alongside so volume-driven exposure and per-case
unreliability stay visibly separate rather than blended into one score.

In [11]:
%%sql
SELECT
    procedure_nm,
    case_count,
    expected_duration_min,
    mae_min,
    pct_outside_30,
    total_deviation_min,
    ROUND(100.0 * total_deviation_min / SUM(total_deviation_min) OVER (), 1) AS pct_of_total_exposure
FROM gold.procedure_baseline
ORDER BY total_deviation_min DESC
LIMIT 20

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 13, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 7 fields>

## Reconciliation check

Confirms the enrichment step didn't drop or duplicate any cases: total deviation summed
directly from `fact_cases` should exactly equal the sum of `total_deviation_min` across
`procedure_baseline`. Any mismatch here would mean the join lost or doubled up rows somewhere
in the enrichment step.

In [12]:
%%sql
SELECT
    (SELECT SUM(abs_error_min) FROM gold.fact_cases) AS fact_total_deviation_min,
    (SELECT SUM(total_deviation_min) FROM gold.procedure_baseline) AS baseline_total_deviation_min

StatementMeta(, f8941938-2381-4fc8-90bd-bb3efa138e21, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

---

# What this notebook establishes

### Final Gold tables

| Table | Rows | Columns |
|---|---:|---:|
| `gold.procedure_baseline` | 418 | 13 |
| `gold.fact_cases` | 48,118 | 26 |

### Answers to the business questions

**1. How reliably do actual durations align with baseline?**

- **54.1 min** mean absolute error (MAE)
- **33.0 min** median absolute error

**2. How often do cases fall outside the expected range?**

- **29.7%** run more than 30 minutes above baseline
- **19.2%** run more than 60 minutes above baseline
- **23.6%** finish more than 30 minutes below baseline
- **46.7%** fall within ±30 minutes of their procedure baseline

**3. What is the estimated annual cost exposure?**

- **452,335 deviation minutes per year**
- **$15.8M/year** at $35 per minute
- **$27.1M/year** at $60 per minute

These are scenario-based estimates of gross exposure, not estimates of fully recoverable
savings.

**4. Which procedures should be prioritized for review?**

Procedures are evaluated using two separate dimensions: **reliability** and **total exposure**.
This prevents a high-volume but relatively predictable procedure from being treated the same
as a genuinely unpredictable one.

The highest-exposure procedure, **Laparotomy, Exploratory**, accounts for only **3.2%** of
total deviation. Exposure is therefore distributed across the procedure cohort rather than
concentrated in a single outlier.

### Reconciliation check

Total deviation from `gold.fact_cases`: **2,600,928 minutes**  
Total deviation from `gold.procedure_baseline`: **2,600,928 minutes**

The totals match exactly, confirming that the enrichment step preserved the full case-level
deviation total without loss or duplication.

### Interpretation boundary

This analysis identifies where historical procedure-duration baselines are least reliable
and quantifies the associated operational and financial exposure.

It does **not** claim that scheduling performance improved or that the estimated dollar
exposure is fully recoverable. No post-intervention period is available to measure such an
outcome.

### Next

Build the **Power BI semantic model** from the two Gold tables.
